In [ ]:
cols = [
    'avg_word_length',
    'avg_sentence_length', 
    'type_token_ratio', 
    'pronoun_freq',
    'punctuation_frequency',
    'info_characters_per_word',
    'info_syll_per_word',
    'info_words_per_sentence',
    'info_type_token_ratio'
]
x_train_df_sliced = x_train_df[cols]

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

In [ ]:
def make_logit_pipeline_poly(degree=1, C=1.0):
    steps = [
        ('poly_features', PolynomialFeatures(degree=degree, include_bias=False)),
        ('rescaler', sklearn.preprocessing.MinMaxScaler()),
        ('logit', sklearn.linear_model.LogisticRegression(solver="lbfgs", l1_ratio=0, C=C, max_iter=1000))
    ]
    pipeline = sklearn.pipeline.Pipeline(steps=steps)
    return pipeline

In [ ]:
degrees = [1, 2, 3, 4, 5, 6, 7]
hypers_list = []

kf = sklearn.model_selection.KFold(n_splits=10, shuffle=True, random_state=42)
for d in degrees:
  pl = make_logit_pipeline_poly(degree=d)
  aucs = []
  for train_index, val_index in kf.split(x_train_df_new):
    x_train_fold = x_train_df_sliced.iloc[train_index]
    y_train_fold = y_train_df['Coarse Label'].iloc[train_index]
    x_val_fold = x_train_df_sliced.iloc[val_index]
    y_val_fold = y_train_df['Coarse Label'].iloc[val_index]

    pl.fit(x_train_fold, y_train_fold)
    y_pred_proba = pl.predict_proba(x_val_fold)[:, 1]

    auc = sklearn.metrics.roc_auc_score(y_val_fold, y_pred_proba)
    aucs.append(auc)
  mean_auc = np.mean(aucs)
  hypers_list.append((d, mean_auc))

AUC_vals = [h[1] for h in hypers_list]
for d, AUC in zip(degrees, AUC_vals):
    print(f"AUC of: {AUC} for degree of: {d}")
plt.plot(degrees, AUC_vals, color='red')
plt.xlabel('Degree')
plt.ylabel('AUCROC')
plt.title('Grid Search of Degree for AUCROC')

best_deg, best_auc = max(hypers_list, key=lambda x: x[1])
print(f"Best degree: {best_deg}, Best AUC: {best_auc}")

In [ ]:
pl_poly = make_logit_pipeline_poly(degree=best_deg)
pl_poly.fit(x_train_df_sliced, y_train_df['Coarse Label'])
y_test_pred_proba = pl_poly.predict_proba(x_test_df[cols])[:, 1]
with open("yproba1_test_p2_best_deg.txt", "w") as f:
    for pred in y_test_pred_proba:
        f.write(f"{pred}\n")

In [ ]:
pl_poly = make_logit_pipeline_poly(degree=3)
pl_poly.fit(x_train_df_sliced, y_train_df['Coarse Label'])
y_test_pred_proba = pl_poly.predict_proba(x_test_df[cols])[:, 1]
with open("yproba1_test_p2_deg_3.txt", "w") as f:
    for pred in y_test_pred_proba:
        f.write(f"{pred}\n")

In [ ]:
C_grid = np.logspace(-4, 4, 17)
hypers_list = []

kf = sklearn.model_selection.KFold(n_splits=10, shuffle=True, random_state=42)
for C in C_grid:
  pl = make_logit_pipeline_poly(degree=2, C=C)
  aucs = []
  for train_index, val_index in kf.split(x_train_df_sliced):
    x_train_fold = x_train_df_sliced.iloc[train_index]
    y_train_fold = y_train_df['Coarse Label'].iloc[train_index]
    x_val_fold = x_train_df_sliced.iloc[val_index]
    y_val_fold = y_train_df['Coarse Label'].iloc[val_index]

    pl.fit(x_train_fold, y_train_fold)
    y_pred_proba = pl.predict_proba(x_val_fold)[:, 1]

    auc = sklearn.metrics.roc_auc_score(y_val_fold, y_pred_proba)
    aucs.append(auc)
  mean_auc = np.mean(aucs)
  hypers_list.append((C, mean_auc))

AUC_vals = [h[1] for h in hypers_list]
for C, AUC in zip(C_grid, AUC_vals):
    print(f"AUC of: {AUC} for C of: {C}")
plt.plot(C_grid, AUC_vals, color='red')
plt.xlabel('C')
plt.ylabel('AUCROC')
plt.title('Grid Search of C for AUCROC')

best_C, best_auc = max(hypers_list, key=lambda x: x[1])
print(f"Best C: {best_C}, Best AUC: {best_auc}")